# Advanced packages - SFR (Streamflow Routing)

Part of the **synthetic-valley advanced-packages** series, which upgrades a calibrated base model's simple boundary conditions to MODFLOW 6 *advanced packages* one notebook at a time. Each notebook loads the shared advanced model from `models/` (creating it from the base model on first use with `mf6_notebook_helpers.load_or_create_advanced_model`) and adds one package, so they can be run independently and in any order - except where a package depends on others.

- **UZF** (`mf6-advanced-packages-uzf`) - recharge (RCH) -> Unsaturated Zone Flow
- **MAW** (`mf6-advanced-packages-maw`) - wells (WEL) -> Multi-Aquifer Well
- **SFR** (`mf6-advanced-packages-sfr`) - river (RIV) -> Streamflow Routing *(this notebook)*
- **LAK** (`mf6-advanced-packages-lak`) - high-K lake -> Lake package
- **MVR** (`mf6-advanced-packages-mvr`) - Water Mover (requires UZF, LAK, SFR)
- **Processing output** (`mf6-advanced-packages-processing`) - run the model and plot results

Replace the river (RIV) package with the Streamflow Routing (SFR) package, which explicitly routes flow between connected stream reaches and exchanges water with the aquifer.

> **Note.** SFR is built *from* the existing RIV package data, so it can only be built once: if the advanced model already contains SFR-1, the build is skipped. Delete the advanced model to rebuild from the base RIV package.

## Imports and setup

Import FloPy and the helpers, then choose the temporal resolution
(`sample_frequency`) and the model name.

In [ ]:
import flopy
import numpy as np
from mf6_notebook_helpers import drop_packages, load_or_create_advanced_model

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

## Load or create the advanced model

Load the shared advanced model from `models/` (or create it from the base model on first use).

In [ ]:
sim = load_or_create_advanced_model(sample_frequency, name)
gwf = sim.get_model(name)

## Build the SFR package

Convert each RIV reach to an SFR reach, connecting the reaches into a single downstream-connected segment and translating the river conductance into a streambed hydraulic conductivity. SFR observations record the surface-water/groundwater exchange and the downstream gauge flow.

In [ ]:
# SFR is built from the RIV package, so build only if SFR-1 is not already present
build_sfr = gwf.get_package("SFR-1") is None
if not build_sfr:
    print(
        "SFR-1 already present - skipping build "
        "(delete the advanced model to rebuild from RIV)"
    )

In [ ]:
if build_sfr:
    pak = gwf.get_package("RIV-1")
    spd = pak.stress_period_data.get_data(0)
    nreaches = len(spd)

    # bed gradient for each reach from the river-bottom elevations
    rgrad = np.diff(spd["rbot"]) * (-1.0)
    rgrad = np.array([float(rgrad[0])] + rgrad.tolist())

    # SFR package data and connection data
    # <ifno> <cellid> <rlen> <rwid> <rgrd> <rtp> <rbth> <rhk> <man> <ncon> <ustrf> <ndv> <boundname>
    delc = gwf.dis.delc.array
    rwid, rbth, man, ustrf, ndv = 30.0, 1.0, 0.03, 1.0, 0
    packagedata = []
    connectiondata = []
    for ifno, (cellid, stage, cond, rbot, boundname) in enumerate(spd):
        rconn = [ifno]
        ncon = 1
        if ifno > 0 and ifno < nreaches - 1:
            ncon += 1
        if ifno > 0:
            rconn.append(ifno - 1)
        if ifno < nreaches - 1:
            rconn.append(-(ifno + 1))
        connectiondata.append(rconn)
        rlen = float(delc[cellid[1]])
        rhk = float(cond) / (rwid * rlen)
        packagedata.append(
            (
                ifno,
                cellid,
                rlen,
                rwid,
                float(rgrad[ifno]),
                float(rbot),
                rbth,
                rhk,
                man,
                ncon,
                ustrf,
                ndv,
                boundname,
            )
        )

    sfr_obs = {
        f"{name}.sfr.obs.csv": [
            ("RIV-SWGW", "SFR", "RIV"),
            ("RIV-FLOW", "downstream-flow", nreaches - 1),
        ]
    }

    sfr = flopy.mf6.ModflowGwfsfr(
        gwf,
        time_conversion=86400.0,
        save_flows=True,
        mover=True,
        boundnames=True,
        nreaches=nreaches,
        packagedata=packagedata,
        connectiondata=connectiondata,
        pname="SFR-1",
    )
    sfr.obs.initialize(filename=f"{name}.sfr.obs", print_input=True, continuous=sfr_obs)

    # remove the river package and its observation package
    drop_packages(gwf, "RIV-1", "RIV_OBS")

gwf.get_package_list()

## Write and run

Write the updated advanced model back to `models/` and run it to confirm the SFR package is valid.

In [ ]:
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
print("SFR advanced model ran successfully")

## Recap

In this notebook you replaced the River (**RIV**) package with the Streamflow Routing (**SFR**) package, which routes flow between connected stream reaches and exchanges water with the aquifer, then wrote the updated advanced model back to `models/` and ran it.